# People counting: multi-model benchmark (CPU / GPU)

Runs several person detectors + ByteTrack on the **same video** and compares them side by side:
people count over time, unique IDs, speed (FPS, ms/frame), GPU memory, and optional accuracy against your manual counts.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (it also runs on CPU, just slower; use a short clip and the `n` model).

**Steps:** run cells top to bottom. Edit the settings in cell 2, then re-run from cell 3.

In [ ]:
# 1. Install and check hardware
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU detected, running on CPU"

In [ ]:
#@title 2. Settings
VIDEO_SOURCE = "sample"   #@param ["sample", "drive", "upload"]
DRIVE_PATH = "/content/drive/MyDrive/sample_videos/room.mp4"  #@param {type:"string"}

# Models to compare (auto-downloaded). Add/remove freely, e.g. "yolo11l.pt", "yolov8s.pt", "rtdetr-x.pt"
MODELS = ["yolo11n.pt", "yolo11s.pt", "yolo11m.pt", "rtdetr-l.pt"]

CONF = 0.3          #@param {type:"number"}
IMGSZ = 640         #@param {type:"integer"}
MAX_SECONDS = 15    #@param {type:"integer"}
VID_STRIDE = 1      #@param {type:"integer"}   # 2 = process every 2nd frame
PRECISION = "auto"  #@param ["auto", "fp16", "fp32"]
OUT_WIDTH = 640     # width of saved result videos (keeps files small)
SHOW_VIDEOS = True  # show every annotated video at the end

# Optional: your own manual counts {seconds_into_video: number_of_people} to get accuracy (MAE)
GROUND_TRUTH = {
    # 2: 3,
    # 5: 4,
    # 8: 4,
}

In [ ]:
# 3. Load the video
import os, time, subprocess
import cv2, torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from base64 import b64encode
from IPython.display import HTML, display
from ultralytics import YOLO, RTDETR

SAMPLE_URL = "https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4"

if VIDEO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO_PATH = DRIVE_PATH
elif VIDEO_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    VIDEO_PATH = "/content/" + next(iter(up))
else:
    VIDEO_PATH = "/content/sample.mp4"
    if not os.path.exists(VIDEO_PATH):
        subprocess.run(["wget", "-q", "-O", VIDEO_PATH, SAMPLE_URL], check=True)

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

scale = min(1.0, OUT_WIDTH / w)
out_w, out_h = int(w * scale) // 2 * 2, int(h * scale) // 2 * 2
max_frames = int(MAX_SECONDS * fps / VID_STRIDE)

has_gpu = torch.cuda.is_available()
device = 0 if has_gpu else "cpu"
use_half = has_gpu if PRECISION == "auto" else (PRECISION == "fp16" and has_gpu)

print(f"Video: {w}x{h} @ {fps:.1f} fps, {total} frames. Processing up to {max_frames} frames.")
print(f"Device: {torch.cuda.get_device_name(0) if has_gpu else 'CPU'} | FP16: {use_half}")

In [ ]:
# 4. Run every model on the video
def load_model(name):
    return RTDETR(name) if name.lower().startswith("rtdetr") else YOLO(name)

def run_model(name):
    tag = name.replace(".pt", "")
    if has_gpu:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = load_model(name)
    # warm-up so first-call overhead doesn't distort timing
    dummy = np.zeros((IMGSZ, IMGSZ, 3), np.uint8)
    for _ in range(2):
        model(dummy, imgsz=IMGSZ, device=device, half=use_half, verbose=False)

    raw_path = f"/content/out_{tag}_raw.mp4"
    final_path = f"/content/out_{tag}.mp4"
    writer = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*"mp4v"), fps / VID_STRIDE, (out_w, out_h))

    counts, ids, infer_ms = [], set(), []
    t0 = time.time()
    stream = model.track(
        source=VIDEO_PATH, stream=True, persist=True, tracker="bytetrack.yaml",
        classes=[0], conf=CONF, imgsz=IMGSZ, device=device, half=use_half,
        vid_stride=VID_STRIDE, verbose=False,
    )
    for i, r in enumerate(stream):
        if i >= max_frames:
            break
        n = len(r.boxes)
        counts.append(n)
        infer_ms.append(r.speed["inference"])
        if r.boxes.id is not None:
            ids.update(r.boxes.id.int().tolist())

        frame = cv2.resize(r.plot(), (out_w, out_h))
        cv2.putText(frame, f"{tag} | in frame: {n} | unique: {len(ids)}",
                    (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        writer.write(frame)
    elapsed = time.time() - t0
    writer.release()

    # re-encode so the browser can play it
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw_path,
                    "-vcodec", "libx264", "-crf", "28", "-pix_fmt", "yuv420p", final_path])

    result = {
        "model": tag,
        "frames": len(counts),
        "avg_count": float(np.mean(counts)),
        "max_count": int(np.max(counts)),
        "unique_ids": len(ids),
        "infer_ms": float(np.mean(infer_ms)),
        "e2e_fps": len(counts) / elapsed,
        "gpu_mem_mb": torch.cuda.max_memory_allocated() / 1e6 if has_gpu else float("nan"),
        "video": final_path,
        "counts": counts,
    }
    del model
    return result

results, failed = [], {}
for name in MODELS:
    print(f"Running {name} ...", end=" ", flush=True)
    try:
        res = run_model(name)
        results.append(res)
        print(f"done ({res['e2e_fps']:.1f} FPS)")
    except Exception as e:
        failed[name] = str(e)
        print(f"FAILED: {e}")

if failed:
    print("\nSkipped models:", failed)

In [ ]:
# 5. Results table
def mae_against_truth(counts, smooth=2):
    errs = []
    for t, truth in GROUND_TRUTH.items():
        i = round(t * fps / VID_STRIDE)
        if i < len(counts):
            window = counts[max(0, i - smooth): i + smooth + 1]
            errs.append(abs(int(np.median(window)) - truth))
    return float(np.mean(errs)) if errs else float("nan")

df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("counts", "video")} for r in results]).set_index("model")
if GROUND_TRUTH:
    df["MAE_people"] = [mae_against_truth(r["counts"]) for r in results]

display(df.round(2))
print("Notes: 'unique_ids' can run high if a person is lost and re-detected. "
      "'infer_ms' is model time only; 'e2e_fps' includes tracking and video writing.")

In [ ]:
# 6. Charts: count over time + speed comparison
fig, ax = plt.subplots(1, 3, figsize=(17, 4))

for r in results:
    t = [i * VID_STRIDE / fps for i in range(len(r["counts"]))]
    ax[0].plot(t, r["counts"], label=r["model"], alpha=0.85)
if GROUND_TRUTH:
    ax[0].scatter(list(GROUND_TRUTH.keys()), list(GROUND_TRUTH.values()),
                  color="black", zorder=5, label="manual count")
ax[0].set_title("People in frame over time"); ax[0].set_xlabel("seconds"); ax[0].legend(); ax[0].grid(alpha=0.3)

df["e2e_fps"].plot.bar(ax=ax[1], color="tab:blue"); ax[1].set_title("End-to-end FPS (higher = faster)"); ax[1].set_xlabel("")
df["avg_count"].plot.bar(ax=ax[2], color="tab:green"); ax[2].set_title("Average people per frame"); ax[2].set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
# 7. Watch the annotated videos
if SHOW_VIDEOS:
    for r in results:
        data = b64encode(open(r["video"], "rb").read()).decode()
        display(HTML(f"<h4>{r['model']}</h4>"
                     f'<video width=560 controls><source src="data:video/mp4;base64,{data}"></video>'))

In [ ]:
# 8. Save results (optional)
df.to_csv("/content/people_counting_results.csv")
print("Saved /content/people_counting_results.csv  (videos are /content/out_<model>.mp4)")

# from google.colab import files
# files.download("/content/people_counting_results.csv")

## Ideas for next experiments
- **CPU vs GPU:** switch the runtime type and re-run; compare the `e2e_fps` column.
- **FP16 vs FP32:** set `PRECISION` to `fp32`, re-run, compare speed and counts.
- **Resolution:** try `IMGSZ = 960` or `1280` if people are small or far away.
- **Accuracy:** fill in `GROUND_TRUTH` with 5-6 manual counts and compare the `MAE_people` column.
- **More models:** add `"yolo11l.pt"`, `"yolov8s.pt"` or `"rtdetr-x.pt"` to `MODELS`.
- **Licensing:** Ultralytics models are AGPL-3.0. For commercial use, consider RF-DETR or another permissively licensed detector.